# CityPulse - Block 0: Exploratory Data Analysis (EDA)

This notebook verifies data integrity, rain event windows, incident clustering, traffic speed vs. `congestion_level` relationships (label leakage check), and per-zone behavioral variations.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Load raw datasets
zones = pd.read_csv('../data/raw/zones.csv')
weather = pd.read_csv('../data/raw/weather.csv')
traffic = pd.read_csv('../data/raw/traffic.csv')
incidents = pd.read_csv('../data/raw/incidents.csv')

weather['timestamp'] = pd.to_datetime(weather['timestamp'])
traffic['timestamp'] = pd.to_datetime(traffic['timestamp'])
incidents['timestamp'] = pd.to_datetime(incidents['timestamp'])

## 1. Data Shapes & Completeness

In [2]:
print(f'Zones: {zones.shape}')
print(f'Weather: {weather.shape}, Missing: {weather.isnull().sum().sum()}')
print(f'Traffic: {traffic.shape}, Missing: {traffic.isnull().sum().sum()}')
print(f'Incidents: {incidents.shape}, Missing: {incidents.isnull().sum().sum()}')

## 2. Rain Event Windows

In [3]:
city_rain = weather.groupby('timestamp')['rainfall_mm'].mean().reset_index()
rain_periods = city_rain[city_rain['rainfall_mm'] > 0]
print(rain_periods)

## 3. Congestion Level vs. Traffic Speed (Leakage Check)

In [4]:
print(traffic.groupby('congestion_level')['avg_speed_kmph'].describe())

## 4. Per-Zone Baseline & Rain Impact

In [5]:
traffic['hourly_timestamp'] = traffic['timestamp'].dt.floor('h')
merged = pd.merge(traffic, weather[['timestamp', 'zone_id', 'rainfall_mm']], 
                  left_on=['hourly_timestamp', 'zone_id'], right_on=['timestamp', 'zone_id'])

zone_perf = merged.groupby('zone_id').agg(
    normal_speed=('avg_speed_kmph', lambda x: x[merged.loc[x.index, 'rainfall_mm'] == 0].mean()),
    rain_speed=('avg_speed_kmph', lambda x: x[merged.loc[x.index, 'rainfall_mm'] > 0].mean())
).reset_index()
zone_perf['speed_drop_%'] = (zone_perf['normal_speed'] - zone_perf['rain_speed']) / zone_perf['normal_speed'] * 100
print(zone_perf)